# C. State preparation from a thermal state with Jaynes-Cummings controls

In [11]:
# ruff: noqa
import os

os.sys.path.append("../../../..")

In [12]:
from feedback_grape.fgrape import optimize_pulse
from feedback_grape.utils.operators import (
    sigmap,
    sigmam,
    create,
    destroy,
    identity,
    cosm,
    sinm,
)
from feedback_grape.utils.states import basis, fock
from feedback_grape.utils.tensor import tensor
import jax.numpy as jnp
import jax
from jax.scipy.linalg import expm

## defining parameterized operations that are repeated num_time_steps times

takes 9 minutes on GPU instead of 17 on CPU

In [13]:
N_cav = 20

In [14]:
def qubit_unitary(alphas):
    alpha_re, alpha_im = alphas
    alpha = alpha_re + 1j * alpha_im
    return tensor(
        identity(N_cav),
        expm(-1j * (alpha * sigmap() + alpha.conjugate() * sigmam()) / 2),
    )

In [15]:
def qubit_cavity_unitary(betas):
    beta_re, beta_im = betas
    beta = beta_re + 1j * beta_im
    return expm(
        -1j
        * (
            beta * (tensor(destroy(N_cav), sigmap()))
            + beta.conjugate() * (tensor(create(N_cav), sigmam()))
        )
        / 2
    )

### povm_measure_operator (callable): <br>
    - It should take a measurement outcome and list of params as input
    - The measurement outcome options are either 1 or -1

In [16]:
from feedback_grape.utils.operators import create, destroy

def povm_measure_operator(measurement_outcome, params):
    """
    POVM for the measurement of the cavity state.
    returns Mm ( NOT the POVM element Em = Mm_dag @ Mm ), given measurement_outcome m, gamma and delta
    """
    gamma, delta = params
    number_operator = tensor(create(N_cav) @ destroy(N_cav), identity(2))
    angle = (gamma * number_operator) + delta / 2
    meas_op = jnp.where(
        measurement_outcome == 1,
        cosm(angle),
        sinm(angle),
    )
    return meas_op

### defining initial (thermal) state

In [17]:
# initial state is a thermal state coupled to a qubit in the ground state?
n_average = 1
# natural logarithm
beta = jnp.log((1 / n_average) + 1)
diags = jnp.exp(-beta * jnp.arange(N_cav))
normalized_diags = diags / jnp.sum(diags, axis=0)
rho_cav = jnp.diag(normalized_diags)

In [18]:
rho_cav.shape

(20, 20)

In [19]:
rho0 = tensor(rho_cav, basis(2, 0) @ basis(2, 0).conj().T)

### defining target state

In [20]:
psi_target = tensor(
    (fock(N_cav, 1) + fock(N_cav, 2) + fock(N_cav, 3)) / jnp.sqrt(3), basis(2)
)
psi_target = psi_target / jnp.linalg.norm(psi_target)

rho_target = psi_target @ psi_target.conj().T
rho_target.shape

(40, 40)

In [21]:
from feedback_grape.utils.fidelity import fidelity

print(fidelity(U_final=rho0, C_target=rho_target, evo_type="density"))

0.14583347808289115


In [22]:
# Print the eigenvalues of rho0
eigenvalues = jnp.linalg.eigvalsh(rho_target)
print("Eigenvalues of rho0:", eigenvalues)

Eigenvalues of rho0: [-1.44095871e-16 -6.66703238e-33  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  5.70266243e-17  1.00000000e+00]


### initialize random params

In [23]:
import jax

print(
    jax.random.uniform(
        jax.random.PRNGKey(0),
        shape=(1, 2),  # 2 for gamma and delta
        minval=-jax.numpy.pi,
        maxval=jax.numpy.pi,
    ).tolist()
)

[[-0.512349077568587, -1.782568231202715]]


In [24]:
import jax
from feedback_grape.fgrape import Gate

num_time_steps = 5
num_of_iterations = 1000
learning_rate = 0.02
# avg_photon_numer = 2 When testing kitten state
# If you provide param_constraints for only one parameter, the current behavior throws an error if you don't provide param_constraints for all parameters for all gates
key1, key2, key3 = jax.random.split(jax.random.PRNGKey(42), 3)
measure = Gate(
    gate=povm_measure_operator,
    initial_params=jax.random.uniform(
        key1,
        shape=(2,),  # 2 for gamma and delta
        minval=-2 * jnp.pi,
        maxval=2 * jnp.pi,
    ),
    measurement_flag=True,
    # param_constraints=[[0, jnp.pi], [-2*jnp.pi, 2*jnp.pi]],
)

qub_unitary = Gate(
    gate=qubit_unitary,
    initial_params=jax.random.uniform(
        key2,
        shape=(2,),  # 2 for gamma and delta
        minval=-2 * jnp.pi,
        maxval=2 * jnp.pi,
    ),
    measurement_flag=False,
    # param_constraints=[[-2*jnp.pi, 2*jnp.pi], [-2*jnp.pi, 2*jnp.pi]],
)

qub_cav = Gate(
    gate=qubit_cavity_unitary,
    initial_params=jax.random.uniform(
        key3,
        shape=(2,),  # 2 for gamma and delta
        minval=-2 * jnp.pi,
        maxval=2 * jnp.pi,
    ),
    measurement_flag=False,
    # param_constraints=[[-jnp.pi, jnp.pi], [-jnp.pi, jnp.pi]],
)

system_params = [measure, qub_unitary, qub_cav]


result = optimize_pulse(
    U_0=rho0,
    C_target=rho_target,
    system_params=system_params,
    num_time_steps=num_time_steps,
    mode="lookup",
    goal="fidelity",
    max_iter=num_of_iterations,
    convergence_threshold=1e-6,
    learning_rate=learning_rate,
    evo_type="density",
    batch_size=10,
)

In [25]:
print(result.final_purity)

[]


In [26]:
#[0.73599227 0.98121761 0.93919395 0.87072329 0.73599227 0.83589782
# 0.98121761 0.93919395 0.93919395 0.98121761]

print(result.final_fidelity)

[0.7316307  0.98108582 0.93893624 0.85876401 0.7316307  0.91122494
 0.98108582 0.93893624 0.93893624 0.98108582]


In [27]:
from feedback_grape.utils.fidelity import fidelity

print(
    "initial fidelity:",
    fidelity(C_target=rho_target, U_final=rho0, evo_type="density"),
)
for i, state in enumerate(result.final_state):
    print(
        f"fidelity of state {i}:",
        fidelity(C_target=rho_target, U_final=state, evo_type="density"),
    )

initial fidelity: 0.14583347808289115
fidelity of state 0: 0.7316307048761646
fidelity of state 1: 0.9810858230866927
fidelity of state 2: 0.9389362396243306
fidelity of state 3: 0.8587640057493503
fidelity of state 4: 0.7316307048761646
fidelity of state 5: 0.9112249421443219
fidelity of state 6: 0.9810858230866927
fidelity of state 7: 0.9389362396243306
fidelity of state 8: 0.9389362396243306
fidelity of state 9: 0.9810858230866927


In [28]:
result.returned_params

[[Array([[-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373],
         [-4.71934159,  0.00485373]], dtype=float64),
  Array([[0.24849854, 2.26995221],
         [3.10119336, 0.55385493],
         [0.24849854, 2.26995221],
         [3.10119336, 0.55385493],
         [0.24849854, 2.26995221],
         [3.10119336, 0.55385493],
         [3.10119336, 0.55385493],
         [0.24849854, 2.26995221],
         [0.24849854, 2.26995221],
         [3.10119336, 0.55385493]], dtype=float64),
  Array([[-1.88634782,  3.42600771],
         [-1.37075863,  1.73481027],
         [-1.88634782,  3.42600771],
         [-1.37075863,  1.73481027],
         [-1.88634782,  3.42600771],
         [-1.37075863,  1.73481027],
         [-1.37075863,  

In [29]:
print(result.iterations)

1000
